# 03: DFT / QFT — Qiskit code lab

このノートブックは `notes/04_discrete_fourier_transform.md`（本文）の **コード実習版** である。

数式や理論の詳細は本文を参照すること。ここでは本文の計算例を Qiskit で再現し、
回路図・状態ベクトル・測定結果を可視化しながら確認する。

### 環境準備

詳細は `notes/00_environment.md` を参照。

```bash
uv venv && source .venv/bin/activate
uv pip install qiskit jupyter matplotlib pylatexenc
jupyter notebook
```

### ライブラリの読み込み

| ライブラリ | 役割 |
|---|---|
| `numpy` | 複素数・行列の数値計算 |
| `matplotlib` | グラフ描画（棒グラフ、ヒストグラム） |
| `QuantumCircuit` | 量子回路の構築（ゲートの追加、回路図の描画） |
| `Statevector` | 量子状態ベクトルの生成・ゲート適用・測定シミュレーション |
| `Operator` | 量子回路からユニタリ行列を数値的に取り出す |

In [ ]:
import numpy as np                          # 数値計算ライブラリ
import matplotlib.pyplot as plt              # グラフ描画ライブラリ
from qiskit import QuantumCircuit            # 量子回路クラス
from qiskit.quantum_info import Statevector  # 量子状態ベクトルクラス
from qiskit.quantum_info import Operator     # ユニタリ行列取得用

# Jupyter 上でグラフをセル内にインライン表示する設定
# これがないとグラフが別ウィンドウで開いてしまう
%matplotlib inline
plt.rcParams['font.size'] = 12  # グラフのフォントサイズを統一

---

## 1. QFT 回路の構成

QFT 回路は3つのステップで構成される:

1. 上位ビットから順にアダマールゲート $H$ を適用
2. より下位のビットから制御位相ゲート $CP(2\pi / 2^m)$ を適用
3. 最後に SWAP でビット順を反転

Qiskit はリトルエンディアン（`q_0` が最下位ビット）なので、
本文の DFT 行列 $F_N$ と一致させるには `q_{n-1}` から処理を開始する。

以下の `qft_circuit(n)` 関数は、任意の量子ビット数 $n$ に対して QFT 回路を構成する。
この関数は以降のセルで繰り返し使用する。

In [ ]:
def qft_circuit(n: int) -> QuantumCircuit:
    """n qubit QFT circuit."""

    # QuantumCircuit(n) で n 量子ビットの空の回路を作成する
    # name は回路図に表示される名前
    qc = QuantumCircuit(n, name=f'QFT({n})')

    # ---- ステップ 1 & 2: H ゲートと制御位相ゲート ----
    #
    # reversed(range(n)) で上位ビット q_{n-1} から q_0 へ順に処理する。
    # 教科書では q_0（最上位）から始める記法が多いが、
    # Qiskit のリトルエンディアン規約に合わせるため逆順にしている。
    for j in reversed(range(n)):

        # qc.h(j): 量子ビット j にアダマールゲートを適用
        # |0> → (|0>+|1>)/sqrt(2), |1> → (|0>-|1>)/sqrt(2)
        qc.h(j)

        # j より下位の各ビット k から制御位相ゲート CP を適用
        # qc.cp(angle, control, target): control ビットが |1> のときだけ
        #   target ビットに位相 e^{i*angle} を付与する
        #
        # 位相角 = 2*pi / 2^(j-k+1)
        # j と k の距離が大きいほど位相角が小さくなる。
        # 例: 隣接ビットなら pi/2、2つ離れていれば pi/4、...
        for k in reversed(range(j)):
            angle = 2 * np.pi / (2 ** (j - k + 1))
            qc.cp(angle, k, j)  # k が制御、j がターゲット

    # ---- ステップ 3: ビット順の反転 (SWAP) ----
    #
    # QFT の数学的な定義では出力のビット順が入力と逆になる。
    # qc.swap(i, j) で量子ビット i と j の状態を入れ替える。
    # 先頭と末尾のペアから順に SWAP していく。
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)

    return qc

---

## 2. $N = 2$（1量子ビット）の QFT

本文で示したとおり $F_2 = H$（アダマールゲート）なので、回路はゲート1つだけになる。

下のセルでは `qft_circuit(1)` で1量子ビットの QFT 回路を生成し、
`.draw('mpl')` で matplotlib を使った回路図を描画する。

In [ ]:
# qft_circuit(1): 1量子ビットの QFT 回路を生成
#   n=1 なので for ループは j=0 の1回だけ → H ゲート1つ
#   SWAP は n//2 = 0 回なので実行されない
#
# .draw('mpl'): matplotlib バックエンドで回路図を描画
#   'mpl' の代わりに 'text' を指定するとテキストベースの簡易表示になる

qft1 = qft_circuit(1)
qft1.draw('mpl')

本文の計算例を再現する。$|0\rangle$ に QFT を適用すると $|+\rangle$ に、$|+\rangle$ に適用すると $|0\rangle$ に戻る。

下のセルの流れ:
1. `Statevector([...])` で入力状態ベクトルを作る
2. `.evolve(qft1)` で QFT 回路を状態に適用する（行列をベクトルに掛ける操作に相当）
3. 出力の各成分の絶対値の2乗（= 測定確率）を棒グラフで表示する

In [ ]:
# ---- 本文「N=2 の計算例」に対応 ----
#
# 計算例 1: F_2 * (1, 0)^T = (1, 1)^T / sqrt(2)
#   |0> が |+> = (|0> + |1>)/sqrt(2) に変換される
#   測定すると |0> と |1> が等確率（各 50%）で出現する
#
# 計算例 2: F_2 * (1, 1)^T / sqrt(2) = (1, 0)^T
#   |+> が |0> に戻る（H は自己逆: H^2 = I）

# 2つの入力を (ラベル, 状態ベクトル) のペアとしてまとめる
cases = [
    ('|0>', Statevector([1, 0])),                            # |0> = (1, 0)^T
    ('|+>', Statevector([1/np.sqrt(2), 1/np.sqrt(2)])),     # |+> = (1, 1)^T / sqrt(2)
]

# 横に2つ並べた棒グラフを作成
# figsize=(幅, 高さ) をインチ単位で指定
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

for ax, (name, sv_in) in zip(axes, cases):
    # .evolve(回路) で量子回路を状態ベクトルに適用する
    # 内部的には回路のユニタリ行列 U を計算し、U * sv_in を返す
    sv_out = sv_in.evolve(qft1)

    # 測定確率 = |振幅|^2 を計算
    # abs(a) は複素数 a の絶対値、**2 で2乗する
    probs = [abs(a)**2 for a in sv_out]

    # 棒グラフを描画
    ax.bar(['|0>', '|1>'], probs, color='coral')
    ax.set_ylabel('Probability')
    ax.set_title(f'QFT({name})')
    ax.set_ylim(0, 1.1)  # 確率の範囲は 0〜1 なので少し余裕を持たせる

plt.tight_layout()  # グラフ同士の間隔を自動調整
plt.show()

---

## 3. $N = 4$（2量子ビット）の QFT

2量子ビットでは $H$、制御位相ゲート $CP(\pi/2)$、SWAP の3つが現れる。

下のセルで回路図を描画し、各ゲートの配置を確認する。

In [ ]:
# qft_circuit(2): 2量子ビットの QFT 回路を生成
#
# n=2 のとき、qft_circuit 内部の処理は以下の順序になる:
#   j=1（上位ビット q_1）: H(1), CP(pi/2, control=0, target=1)
#   j=0（下位ビット q_0）: H(0)
#   SWAP(0, 1)
#
# 回路図では左から右へ時間が進む。
# 縦線で結ばれたゲートは制御ゲート（上の丸が制御ビット）。

qft2 = qft_circuit(2)
qft2.draw('mpl')

本文の3つの計算例を再現する。

下のセルでは3つの入力を一括で処理し、3行×2列のグラフにまとめて表示する。
各行は1つの計算例に対応し、左列が入力の振幅、右列が QFT 後の測定確率を示す。

In [ ]:
# ---- 本文「N=4 の計算例 1〜3」に対応 ----
#
# 計算例 1: 一様分布 (1/2)(1,1,1,1)^T → (1,0,0,0)^T
#   振動なし（一定値）なので周波数 0（DC 成分）のみが残る
#
# 計算例 2: 交互パターン (1/2)(1,-1,1,-1)^T → (0,0,1,0)^T
#   2サンプルごとに1周期 = 最高周波数 k=2 (= N/2) の成分のみ
#
# 計算例 3: 単一成分 (1,0,0,0)^T → (1/2)(1,1,1,1)^T
#   離散デルタ関数の DFT は全周波数に一様分布（計算例 1 と対称）

# 3つの入力を (タイトル, ベクトル) のリストで定義
# ベクトルは Python のリストで与え、Statevector() で量子状態に変換する
examples = [
    ('Ex.1: Uniform',      [0.5, 0.5, 0.5, 0.5]),      # 一様: 全成分が 1/2
    ('Ex.2: Alternating',  [0.5, -0.5, 0.5, -0.5]),    # 交互: 符号が +, -, +, -
    ('Ex.3: Single',       [1, 0, 0, 0]),               # 単一: j=0 のみ非零
]

# 2量子ビットの計算基底ラベル
# |00> = j=0, |01> = j=1, |10> = j=2, |11> = j=3
labels = ['|00>', '|01>', '|10>', '|11>']

# 3行 × 2列のサブプロットを作成
# axes[行][列] で個別のグラフ領域（Axes オブジェクト）にアクセスする
fig, axes = plt.subplots(3, 2, figsize=(10, 9))

for row, (title, vec) in zip(axes, examples):
    sv_in = Statevector(vec)         # リストから量子状態ベクトルを生成
    sv_out = sv_in.evolve(qft2)      # QFT 回路を適用

    # ---- 左列: 入力の振幅（実部）----
    # 交互パターンのように負の振幅を持つ入力があるため、
    # 確率（|a|^2、常に正）ではなく振幅の実部を表示する。
    # 正の振幅を青、負の振幅を赤で色分けする。
    amps = [a.real for a in sv_in]   # .real で複素数の実部を取得
    colors = ['steelblue' if a >= 0 else 'salmon' for a in amps]
    row[0].bar(labels, amps, color=colors)
    row[0].set_ylabel('Amplitude')
    row[0].set_title(f'{title} (input)')
    row[0].set_ylim(-0.7, 1.1)
    row[0].axhline(0, color='k', linewidth=0.5)  # y=0 に基準線を引く

    # ---- 右列: QFT 後の測定確率 ----
    # 測定確率 = |振幅|^2。複素振幅でも絶対値の2乗は常に正の実数。
    probs = [abs(a)**2 for a in sv_out]
    row[1].bar(labels, probs, color='coral')
    row[1].set_ylabel('Probability')
    row[1].set_title(f'{title} (after QFT)')
    row[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

---

## 4. 測定シミュレーション

量子コンピュータでは状態ベクトルを直接読むことはできない。
測定のたびに確率的に1つの計算基底が得られるだけである。
多数回の測定で出現頻度から確率分布を推定する。

下のセルでは `.sample_counts(1024)` を使い、1024回の測定をシミュレーションする。
上のセクション3で見た確率分布と、ここでの出現頻度が対応していることを確認する。

- 確率 1.0 の状態 → 1024回中ほぼ全回その状態が出現
- 確率 0.25 × 4状態 → 各状態が約 256 回ずつ出現（統計的なばらつきあり）

In [ ]:
# 3つの計算例について QFT 後の状態を 1024 回測定する

# 横に3つ並べたサブプロット
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (title, vec) in zip(axes, examples):
    # QFT を適用
    sv_out = Statevector(vec).evolve(qft2)

    # .sample_counts(shots) で測定を shots 回シミュレーションする
    # 戻り値は辞書: {'00': 回数, '01': 回数, ...}
    # 確率 0 の状態はキーに含まれないことがある
    counts = sv_out.sample_counts(1024)

    # すべての計算基底について出現回数を取得
    # counts.get(l, 0) はキーが存在しなければ 0 を返す
    all_labels = ['00', '01', '10', '11']
    values = [counts.get(l, 0) for l in all_labels]

    ax.bar(all_labels, values, color='coral')
    ax.set_xlabel('Measurement result')
    ax.set_ylabel('Counts / 1024')
    ax.set_title(title)
    ax.set_ylim(0, 1100)

plt.suptitle('Measurement after QFT (1024 shots)', fontsize=14)
plt.tight_layout()
plt.show()

---

## 5. QFT 回路と DFT 行列の一致検証

本文で定義した DFT 行列 $(F_N)_{kj} = \frac{1}{\sqrt{N}} \omega_N^{jk}$ を NumPy で構成し、
QFT 回路のユニタリ行列と数値的に一致するか確認する。

下のセルの流れ:
1. `Operator(回路).data` で QFT 回路のユニタリ行列を NumPy 配列として取得する
2. 本文の定義式に従い DFT 行列 $F_N$ を NumPy で構成する
3. `np.allclose()` で2つの行列が数値的に一致するか判定する

In [ ]:
# N = 2, 4, 8 の3つのサイズで検証する

for n in [1, 2, 3]:
    N = 2 ** n  # N = 2^n: 状態空間の次元数

    # ---- QFT 回路からユニタリ行列を取得 ----
    # Operator(回路) は回路全体を1つのユニタリ行列として計算する
    # .data で N×N の NumPy 複素数配列を取り出す
    U_qft = Operator(qft_circuit(n)).data

    # ---- DFT 行列を本文の定義どおりに構成 ----
    # omega_N = e^{2*pi*i / N}: 1 の N 乗根
    omega = np.exp(2j * np.pi / N)

    # (F_N)_{kj} = (1/sqrt(N)) * omega_N^{j*k}
    # 外側のリスト内包表記が行 (k)、内側が列 (j) に対応
    F_N = np.array([
        [omega ** (j * k) for j in range(N)]  # k 行目: j=0,1,...,N-1
        for k in range(N)                      # k=0,1,...,N-1
    ]) / np.sqrt(N)  # 全体に 1/sqrt(N) を掛ける

    # ---- 数値的な一致を判定 ----
    # np.allclose(A, B) は全要素について |A_{ij} - B_{ij}| < 1e-8 なら True
    match = np.allclose(U_qft, F_N)
    print(f'N={N} (n={n} qubits): QFT circuit = F_{N}? -> {match}')